In [0]:
%run /Repos/dung_nguyen_hoang@mfcgd.com/Utilities/Functions

In [0]:
import pyspark.sql.functions as F

lst_mthend = '2025-07-31'

df_path = "/mnt/lab/vn/project/VN_NB/CSV_FILES/minor_claim_client_list.csv"

df = spark.read.csv(df_path, header=True, inferSchema=True)

df = df.withColumn(
    "cli_num", 
    F.col("cli_num").cast("string")
).withColumn(
    "image_date",
    F.lit(lst_mthend).cast("date")
)

In [0]:
cli_base_benefits = spark.sql(f"""
WITH BENEFITS AS (
SELECT  pol.image_date,
        --pol.POL_NUM, 
        cvg.CLI_NUM,
        CONCAT_WS('/',
        COLLECT_SET(pol.POL_NUM)) AS POLICIES,
        CONCAT_WS('/',
        COLLECT_SET(TO_DATE(pol.POL_ISS_DT))) AS ISSUE_DATES, 
        CONCAT_WS('/', 
        COLLECT_SET(CASE WHEN cvg.CVG_TYP='B' THEN pol.PLAN_CODE_BASE END)) AS BENEFIT_BASE
FROM    vn_published_casm_cas_snapshot_db.tcoverages cvg
   INNER JOIN
        vn_published_casm_cas_snapshot_db.tpolicys pol ON
        (cvg.POL_NUM = pol.POL_NUM AND cvg.PLAN_CODE = pol.PLAN_CODE_BASE AND cvg.image_date = pol.image_date)
   INNER JOIN 
        vn_published_cas_db.tplans pln ON 
        (cvg.plan_code=pln.plan_code AND cvg.vers_num=pln.vers_num)
WHERE   cvg.image_date = '{lst_mthend}'
    AND cvg.CVG_TYP = 'B'
    AND pol.POL_STAT_CD IN ('1','2','3','5','7','9')
    AND pol.POL_ISS_DT BETWEEN '2025-01-01' AND '{lst_mthend}'
GROUP BY
        pol.image_date, --pol.POL_NUM, 
        cvg.CLI_NUM
) 
SELECT  DISTINCT 
        image_date, 
        CLI_NUM, 
        POLICIES, BENEFIT_BASE, ISSUE_DATES,
        'Y' AS MAIN_LA_IND
FROM    BENEFITS                               
""")

# check_dup(cli_base_benefits, "CLI_NUM")

In [0]:
#cli_base_benefits.createOrReplaceTempView("pol_base")
cli_rider_benefits = spark.sql(f"""
WITH POL_BASE AS(
SELECT  POL_NUM, image_date
FROM    vn_published_casm_cas_snapshot_db.tpolicys
WHERE   image_date = '{lst_mthend}'
    AND POL_STAT_CD IN ('1','2','3','5','7','9')
    AND POL_ISS_DT BETWEEN '2025-01-01' AND '{lst_mthend}'
), BENEFITS AS (
SELECT  cvg.image_date,
        --cvg.POL_NUM, 
        cvg.CLI_NUM,        
        CONCAT_WS('/', 
        COLLECT_SET(CASE WHEN cvg.CVG_TYP='R' THEN PROD_TYP END)) AS BENEFIT_RIDER
FROM    pol_base pol
   INNER JOIN 
        vn_published_casm_cas_snapshot_db.tcoverages cvg ON
        (pol.POL_NUM = cvg.POL_NUM AND pol.image_date = cvg.image_date)
   INNER JOIN
        vn_published_cas_db.tplans pln ON 
        (cvg.plan_code=pln.plan_code AND cvg.vers_num=pln.vers_num)
WHERE   cvg.CVG_TYP = 'R'
    AND cvg.CVG_STAT_CD IN ('1','2','3','5','7','9')
GROUP BY
        cvg.image_date, --cvg.POL_NUM, 
        cvg.CLI_NUM
) 
SELECT  image_date, --POL_NUM, 
        CLI_NUM, BENEFIT_RIDER,
        'Y' AS OTHER_INSURED_IND
FROM    BENEFITS                               
""")

# check_dup(cli_rider_benefits, "CLI_NUM")

In [0]:
result = df.join(
    cli_base_benefits,
    on=["cli_num","image_date"],
    how="inner"
).join(
    cli_rider_benefits,
    on=["cli_num","image_date"],
    how="inner"
).fillna(
    {
        "BENEFIT_BASE": "NONE", "MAIN_LA_IND": "N",
        "BENEFIT_RIDER": "NONE", "OTHER_INSURED_IND": "N"
    }
)

#print(result.count())

In [0]:
final = result.toPandas()
print(final.shape)
final.to_csv(
    "/dbfs/mnt/lab/vn/project/VN_NB/CSV_FILES/minor_claim_client_new_sales.csv",
    index=False,
    header=True)